In [ ]:
import torch
import numpy as np
import os
import re
from transformers import BertTokenizer, BertModel
from typing import Dict, List
import pickle

# 配置路径
MODEL_PATH = "bert-base-chinese-local"  # 本地BERT模型路径
RAG_DIR = "rag"  # TXT文件目录
VECTOR_DB_PATH = "vector_db.pkl"  # 向量库保存路径
CHUNK_SIZE = 512  # 分块大小（字符数）
OVERLAP_SIZE = 50  # 块之间的重叠字符数

# 初始化本地BERT模型和分词器
print("加载本地BERT模型...")
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertModel.from_pretrained(MODEL_PATH)
model.eval()  # 设置为评估模式


def read_txt_files(txt_dir: str) -> Dict[str, str]:
    """读取所有TXT文件，返回字典：{文件名: 内容}"""
    txt_contents = {}
    for filename in os.listdir(txt_dir):
        if filename.endswith(".txt"):
            filepath = os.path.join(txt_dir, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                txt_contents[filename] = content
    print(f"共读取 {len(txt_contents)} 个TXT文件")
    return txt_contents


def split_text_into_chunks(text: str, chunk_size: int = CHUNK_SIZE, overlap_size: int = OVERLAP_SIZE) -> List[str]:
    """
    将文本分割成固定大小的块
    保持句子完整性，避免在句子中间切分
    """
    # 按句子分割（中文标点）
    sentences = re.split(r'([。！？；\.\?!;])', text)

    # 重新组合句子，保持标点
    sentences_with_punct = []
    for i in range(0, len(sentences) - 1, 2):
        if i + 1 < len(sentences):
            sentences_with_punct.append(sentences[i] + sentences[i + 1])
        else:
            sentences_with_punct.append(sentences[i])

    if len(sentences) % 2 == 1:  # 处理最后一个单独的句子
        sentences_with_punct.append(sentences[-1])

    # 按句子合并成块
    chunks = []
    current_chunk = ""

    for sentence in sentences_with_punct:
        # 如果当前句子加上当前块超过chunk_size，则保存当前块
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # 保留重叠部分
            overlap_start = max(0, len(current_chunk) - overlap_size)
            current_chunk = current_chunk[overlap_start:] + sentence
        else:
            current_chunk += sentence

    # 添加最后一个块
    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def get_bert_embedding(text: str, max_length: int = 512) -> np.ndarray:
    """使用BERT获取文本的嵌入向量（取CLS token的表示）"""
    # 分词和编码
    inputs = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="pt"
    )

    # 推理
    with torch.no_grad():
        outputs = model(**inputs)
        # 取CLS token的表示（最后一层隐藏状态）
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

    return cls_embedding


def process_document_chunks(filename: str, content: str) -> List[Dict]:
    """处理单个文档，将其分块并生成向量"""
    # 分块
    chunks = split_text_into_chunks(content, CHUNK_SIZE, OVERLAP_SIZE)
    print(f"  '{filename}' 分割为 {len(chunks)} 个块")

    # 为每个块生成向量
    chunk_data = []
    for i, chunk in enumerate(chunks):
        # 生成向量
        embedding = get_bert_embedding(chunk)

        # 存储块信息
        chunk_info = {
            "chunk_id": i,
            "chunk_text": chunk,
            "embedding": embedding,
            "embedding_dim": embedding.shape[-1],
            "chunk_length": len(chunk),
            "parent_doc": filename,
            "chunk_range": f"{i + 1}/{len(chunks)}"
        }
        chunk_data.append(chunk_info)

    return chunk_data


def build_vector_db(txt_contents: Dict[str, str]) -> Dict[str, dict]:
    """构建向量数据库，返回字典结构"""
    vector_db = {}
    total_chunks = 0

    for idx, (filename, content) in enumerate(txt_contents.items()):
        print(f"处理文件 [{idx + 1}/{len(txt_contents)}]: {filename}")

        # 处理文档分块
        chunks_data = process_document_chunks(filename, content)
        total_chunks += len(chunks_data)

        # 存储到向量库
        vector_db[filename] = {
            "filename": filename,
            "original_content": content,
            "total_chunks": len(chunks_data),
            "chunks": chunks_data,
            "metadata": {
                "original_length": len(content),
                "avg_chunk_length": sum(c["chunk_length"] for c in chunks_data) / len(chunks_data) if chunks_data else 0
            }
        }

    print(f"\n总计处理 {len(vector_db)} 个文档，{total_chunks} 个文本块")
    return vector_db


def save_vector_db(vector_db: Dict[str, dict], save_path: str):
    """保存向量数据库到文件"""
    with open(save_path, 'wb') as f:
        pickle.dump(vector_db, f)
    print(f"向量库已保存到: {save_path}")


if __name__ == "__main__":
    # 1. 读取TXT文件
    print("步骤1: 读取TXT文件...")
    txt_contents = read_txt_files(RAG_DIR)

    # 2. 构建向量库（带分块）
    print("\n步骤2: 构建向量库（分块处理）...")
    print(f"分块参数: chunk_size={CHUNK_SIZE}, overlap={OVERLAP_SIZE}")
    vector_db = build_vector_db(txt_contents)

    # 3. 保存向量库
    print("\n步骤3: 保存向量库...")
    save_vector_db(vector_db, VECTOR_DB_PATH)

    # 4. 输出统计信息
    print("\n" + "=" * 60)
    print("向量库构建完成！")
    print(f"文档数量: {len(vector_db)}")

    # 计算总文本块数
    total_chunks = sum(doc["total_chunks"] for doc in vector_db.values())
    print(f"文本块总数: {total_chunks}")